# 예제 02. 마지막 계층 교체하기
빅데이터프로그래밍 · 10주차

## 목표
- 기존 출력 클래스 수를 확인한다
- 새 데이터셋에 맞춰 마지막 계층을 바꾼다
- 바꾼 뒤 출력 shape을 확인한다

전이학습에서 실제로 손대는 코드는 **한 줄**입니다.


In [ ]:
import torch
import torch.nn as nn
from torchvision import models

device = "cuda" if torch.cuda.is_available() else "cpu"
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)


## 1. 바꾸기 전 확인


In [ ]:
print("현재 fc:", model.fc)
print("in_features :", model.fc.in_features)
print("out_features:", model.fc.out_features)

x = torch.randn(2, 3, 224, 224)
print("\n출력 shape:", tuple(model(x).shape), "← 1,000개 클래스")


## 2. 교체 — 한 줄입니다
`in_features` 는 그대로 두고 출력만 우리 클래스 수로 바꿉니다.


In [ ]:
N_CLASSES = 5                                # 꽃 5종류

model.fc = nn.Linear(model.fc.in_features, N_CLASSES)

print("바꾼 fc:", model.fc)
print("출력 shape:", tuple(model(x).shape), "← 5개 클래스")


## 3. in_features 를 직접 쓰면 위험합니다
모델마다 값이 다릅니다. 하드코딩하지 말고 `model.fc.in_features` 를 읽으세요.


In [ ]:
m18 = models.resnet18(weights=None)
m50 = models.resnet50(weights=None)
print("resnet18 in_features:", m18.fc.in_features)
print("resnet50 in_features:", m50.fc.in_features, "← 다릅니다")

# resnet50 에 512를 쓰면
try:
    m50.fc = nn.Linear(512, 5)
    m50(torch.randn(1, 3, 224, 224))
except RuntimeError as err:
    print("\nRuntimeError:", err)


In [ ]:
# 올바른 방법
m50.fc = nn.Linear(m50.fc.in_features, 5)
print("해결:", tuple(m50(torch.randn(1, 3, 224, 224)).shape))


## 4. 새 계층은 무작위로 초기화됩니다
교체한 `fc` 만 처음부터 배웁니다. 앞부분은 이미 배운 상태입니다.


In [ ]:
model2 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
old_fc = model2.fc.weight.data.clone()
model2.fc = nn.Linear(model2.fc.in_features, 5)

print("기존 fc 가중치 평균:", old_fc.mean().item())
print("새   fc 가중치 평균:", model2.fc.weight.data.mean().item(), "← 무작위")


## 5. MobileNet은 이름이 다릅니다


In [ ]:
mob = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
print("교체 전:", mob.classifier)

mob.classifier[-1] = nn.Linear(mob.classifier[-1].in_features, 5)
print("\n교체 후:", mob.classifier)
print("출력:", tuple(mob(torch.randn(2, 3, 224, 224)).shape))


## 6. 계층을 두 개로 늘려도 됩니다
꼭 Linear 하나일 필요는 없습니다. Dropout을 끼워 넣는 것도 흔합니다.


In [ ]:
model3 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
in_f = model3.fc.in_features

model3.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_f, 256),
    nn.ReLU(),
    nn.Linear(256, N_CLASSES),
)
print(model3.fc)
print("출력:", tuple(model3(x).shape))


## 직접 해보기
1. `resnet34` 의 마지막 계층을 클래스 10개로 교체하고 출력 shape을 확인하세요.
2. `efficientnet_b0` 의 마지막 계층 이름은 무엇인가요? 교체해 보세요.


In [ ]:
# 여기에 작성하세요
